In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load the official 2026 dataset
url = (
    "https://raw.githubusercontent.com/"
    "DataTalksClub/machine-learning-zoomcamp/main/"
    "cohorts/2026/data/course_lead_scoring_2026.csv"
)
df = pd.read_csv(url)

# 2. Prepare the data
target = df["converted"]
features = df.drop(columns=["converted"]).copy()

categorical = features.select_dtypes(include=["object"]).columns
numerical = features.select_dtypes(exclude=["object"]).columns

features[categorical] = features[categorical].fillna("NA")
features[numerical] = features[numerical].fillna(0.0)

data = features.copy()
data["converted"] = target

# 3. Exact train, validation and test split
df_full_train, df_test = train_test_split(
    data, test_size=0.2, random_state=1
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train["converted"]
y_val = df_val["converted"]

X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])

# 4. Question 1: ROC AUC feature importance
print("QUESTION 1")
auc_features = {}

for col in numerical:
    auc = roc_auc_score(y_train, X_train[col])
    auc_features[col] = max(auc, 1 - auc)

candidates = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]
print("Best feature:", max(candidates, key=lambda c: auc_features[c]))

# 5. Train the logistic regression model
dv = DictVectorizer(sparse=False)

X_train_matrix = dv.fit_transform(
    X_train.to_dict(orient="records")
)
X_val_matrix = dv.transform(
    X_val.to_dict(orient="records")
)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train_matrix, y_train)

probabilities = model.predict_proba(X_val_matrix)[:, 1]

# 6. Question 2: Model AUC
print("\nQUESTION 2")
auc = roc_auc_score(y_val, probabilities)
print("Validation AUC:", round(auc, 3))

# 7. Question 3: Precision and recall intersection
print("\nQUESTION 3")
thresholds = np.arange(0.0, 1.0, 0.01)
differences = []

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)
    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    if precision == 0 and recall == 0:
        continue

    differences.append(
        (abs(precision - recall), threshold, precision, recall)
    )

best_intersection = min(differences)
print("Threshold:", round(best_intersection[1], 2))
print("Precision:", round(best_intersection[2], 3))
print("Recall:", round(best_intersection[3], 3))

# 8. Question 4: Threshold with maximum F1
print("\nQUESTION 4")
f1_scores = []

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)
    f1 = f1_score(y_val, predictions, zero_division=0)
    f1_scores.append(f1)

best_threshold = thresholds[np.argmax(f1_scores)]
print("Best F1 threshold:", round(best_threshold, 2))
print("Maximum F1:", round(max(f1_scores), 3))

# 9. Questions 5 and 6: Five-fold cross-validation
print("\nQUESTIONS 5 AND 6")
cv = KFold(n_splits=5, shuffle=True, random_state=1)

def evaluate_c(c):
    scores = []

    for train_idx, val_idx in cv.split(df_full_train):
        fold_train = df_full_train.iloc[train_idx].copy()
        fold_val = df_full_train.iloc[val_idx].copy()

        fold_y_train = fold_train.pop("converted")
        fold_y_val = fold_val.pop("converted")

        vectorizer = DictVectorizer(sparse=False)
        fold_train_matrix = vectorizer.fit_transform(
            fold_train.to_dict(orient="records")
        )
        fold_val_matrix = vectorizer.transform(
            fold_val.to_dict(orient="records")
        )

        clf = LogisticRegression(
            solver="liblinear",
            C=c,
            max_iter=1000,
            random_state=42
        )
        clf.fit(fold_train_matrix, fold_y_train)

        fold_probabilities = clf.predict_proba(
            fold_val_matrix
        )[:, 1]

        scores.append(
            roc_auc_score(fold_y_val, fold_probabilities)
        )

    return np.array(scores)

# Question 5
scores = evaluate_c(1.0)
print("Q5 - Fold AUC scores:", np.round(scores, 3))
print("Q5 - Standard deviation:", round(np.std(scores), 3))

# Question 6
print("\nQ6 - Hyperparameter tuning")
tuning = {}

for c in [0.000001, 0.001, 1.0]:
    scores = evaluate_c(c)
    tuning[c] = (np.mean(scores), np.std(scores))
    print(
        f"C={c:g}: mean AUC={np.mean(scores):.3f}, "
        f"std={np.std(scores):.3f}"
    )

best_c = max(
    tuning,
    key=lambda c: (
        round(tuning[c][0], 3),
        -round(tuning[c][1], 3),
        -c
    )
)
print("Best C:", best_c)


QUESTION 1
Best feature: lead_score

QUESTION 2
Validation AUC: 0.732

QUESTION 3
Threshold: 0.63
Precision: 0.733
Recall: 0.725

QUESTION 4
Best F1 threshold: 0.41
Maximum F1: 0.758

QUESTIONS 5 AND 6
Q5 - Fold AUC scores: [0.731 0.73  0.736 0.741 0.721]
Q5 - Standard deviation: 0.007

Q6 - Hyperparameter tuning
C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007
Best C: 0.001
